In [1]:
%%writefile vector_add.cu

#include <stdio.h>
#include <stdlib.h>

// ============================================
// STEP 1: GPU KERNEL FUNCTION
// This runs ON the GPU — each thread adds ONE element
// ============================================
__global__ void vectorAdd(int *A, int *B, int *C, int N)
{
    // Each thread finds its unique position (index)
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Safety check: only work if index is within array size
    if (i < N)
    {
        C[i] = A[i] + B[i];  // Thread i adds A[i] + B[i] and stores in C[i]
    }
}

// ============================================
// STEP 2: MAIN FUNCTION (runs on CPU)
// ============================================
int main()
{
    int N;

    // ------- Take input from user -------
    printf("Enter the size of vectors: ");
    scanf("%d", &N);

    // ------- Allocate memory on CPU (Host) -------
    int *A = (int*)malloc(N * sizeof(int));  // Array A on CPU
    int *B = (int*)malloc(N * sizeof(int));  // Array B on CPU
    int *C = (int*)malloc(N * sizeof(int));  // Result array on CPU

    // ------- Take vector elements from user -------
    printf("Enter %d elements for Vector A:\n", N);
    for (int i = 0; i < N; i++)
    {
        printf("A[%d] = ", i);
        scanf("%d", &A[i]);
    }

    printf("Enter %d elements for Vector B:\n", N);
    for (int i = 0; i < N; i++)
    {
        printf("B[%d] = ", i);
        scanf("%d", &B[i]);
    }

    // ------- Declare GPU (Device) pointers -------
    int *d_A, *d_B, *d_C;   // d_ means "device" (GPU)

    // ------- Allocate memory ON the GPU -------
    cudaMalloc(&d_A, N * sizeof(int));
    cudaMalloc(&d_B, N * sizeof(int));
    cudaMalloc(&d_C, N * sizeof(int));

    // ------- Copy data from CPU --> GPU -------
    cudaMemcpy(d_A, A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N * sizeof(int), cudaMemcpyHostToDevice);

    // ------- Set up blocks and threads -------
    int threadsPerBlock = 256;  // Each block has 256 threads
    // How many blocks do we need?
    // If N=1000, threads=256: we need ceil(1000/256) = 4 blocks
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // ------- LAUNCH the GPU kernel! -------
    // This starts ALL threads on the GPU simultaneously
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // ------- Copy result from GPU --> CPU -------
    cudaMemcpy(C, d_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    // ------- Print the result -------
    printf("\nVector A: ");
    for (int i = 0; i < N; i++) printf("%d ", A[i]);

    printf("\nVector B: ");
    for (int i = 0; i < N; i++) printf("%d ", B[i]);

    printf("\nResult C = A + B: ");
    for (int i = 0; i < N; i++) printf("%d ", C[i]);
    printf("\n");

    // ------- Free all memory -------
    free(A); free(B); free(C);           // Free CPU memory
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);  // Free GPU memory

    return 0;
}

Writing vector_add.cu


In [2]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!echo -e "5\n1 2 3 4 5\n10 20 30 40 50" | ./vector_add

Enter the size of vectors: Enter 5 elements for Vector A:
A[0] = A[1] = A[2] = A[3] = A[4] = Enter 5 elements for Vector B:
B[0] = B[1] = B[2] = B[3] = B[4] = 
Vector A: 1 2 3 4 5 
Vector B: 10 20 30 40 50 
Result C = A + B: 11 22 33 44 55 
